In [3]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")


CUDA available: True
Device: Tesla T4


In [4]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [5]:
import sys
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/image_realness_project")
sys.path.insert(0, str(PROJECT))

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


In [6]:
from core.models.joint_model import JointModel

joint_model = JointModel(use_pretrained_backbones=True).to(device)
state = torch.load(PROJECT / "checkpoints" / "JOINT_2024.pth", map_location=device)
joint_model.load_state_dict(state)
joint_model.eval()
print("JOINT loaded")

Downloading: "https://download.pytorch.org/models/swin_t-704ceda3.pth" to /root/.cache/torch/hub/checkpoints/swin_t-704ceda3.pth


100%|██████████| 108M/108M [00:00<00:00, 185MB/s] 


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 110MB/s]


JOINT loaded


In [7]:
from core.guidance.rationality_guidance import RationalityGuidanceScorer

scorer = RationalityGuidanceScorer(joint_model, image_size=224, blur_kernel=11).to(device)
scorer.eval()

RationalityGuidanceScorer(
  (joint_model): JointModel(
    (technical_feature_extraction): SwinTransformer(
      (features): Sequential(
        (0): Sequential(
          (0): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
          (1): Permute()
          (2): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
        )
        (1): Sequential(
          (0): SwinTransformerBlock(
            (norm1): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
            (attn): ShiftedWindowAttention(
              (qkv): Linear(in_features=96, out_features=288, bias=True)
              (proj): Linear(in_features=96, out_features=96, bias=True)
            )
            (stochastic_depth): StochasticDepth(p=0.0, mode=row)
            (norm2): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
            (mlp): MLP(
              (0): Linear(in_features=96, out_features=384, bias=True)
              (1): GELU(approximate='none')
              (2): Dropout(p=0.0, inplace=False)
  

In [8]:
from core.generation.sd_generator import load_sd_pipeline

pipe = load_sd_pipeline(project=PROJECT, device=device.type)
pipe.safety_checker = None   # skip for speed
print("SD loaded")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

SD loaded


In [9]:

PROMPTS = [
    "a golden retriever in a sunlit meadow",
    "a bowl of fresh fruit on a wooden table",
    "a mountain lake at sunset with reflections",
    "a busy street market in the morning",
    "a cat sitting on a windowsill in rain",
]

HYPERPARAM_GRID = [
    {"rationality_weight": 0.0005, "guidance_last_steps": 5},
    {"rationality_weight": 0.002,  "guidance_last_steps": 5},
    {"rationality_weight": 0.003,  "guidance_last_steps": 15},
    {"rationality_weight": 0.01,  "guidance_last_steps": 20}
]

SEED               = 42
NUM_STEPS          = 30
GUIDANCE_SCALE     = 7.5
EXPERIMENT_DIR     = PROJECT / "outputs" / "experiment_grid"
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

In [10]:
import subprocess, sys, csv
from core.generation.sd_generator import generate_image

rows = []  # will collect all metadata for scoring later

for prompt in PROMPTS:
    # sanitize prompt to use as folder name
    prompt_slug = prompt[:40].replace(" ", "_").replace(",", "")
    prompt_dir  = EXPERIMENT_DIR / prompt_slug
    prompt_dir.mkdir(exist_ok=True)

    # --- without guidance ---
    no_guidance_path = prompt_dir / "no_guidance.png"
    generate_image(
        pipe=pipe,
        prompt=prompt,
        output_path=no_guidance_path,
        guidance_scale=GUIDANCE_SCALE,
        num_inference_steps=NUM_STEPS,
        seed=SEED,
    )
    rows.append({
        "prompt":            prompt,
        "mode":              "no_guidance",
        "rationality_weight": None,
        "guidance_last_steps": None,
        "image_path":        str(no_guidance_path),
    })
    print(f"[no_guidance] saved: {no_guidance_path.name}")

    # --- with guidance, each hyperparam combo ---
    for hp in HYPERPARAM_GRID:
        run_name  = f"rw{hp['rationality_weight']}_ls{hp['guidance_last_steps']}"
        guided_dir = prompt_dir / run_name
        guided_dir.mkdir(exist_ok=True)

        subprocess.run([
            sys.executable,
            str(PROJECT / "scripts" / "generate_with_rationality_guidance.py"),
            "--project",             str(PROJECT),
            "--prompt",              prompt,
            "--output-dir",          str(guided_dir),
            "--num-inference-steps", str(NUM_STEPS),
            "--guidance-scale",      str(GUIDANCE_SCALE),
            "--rationality-weight",  str(hp["rationality_weight"]),
            "--guidance-last-steps", str(hp["guidance_last_steps"]),
            "--seed",                str(SEED),
        ])

        rows.append({
            "prompt":             prompt,
            "mode":               "guided",
            "rationality_weight": hp["rationality_weight"],
            "guidance_last_steps": hp["guidance_last_steps"],
            "image_path":         str(guided_dir / "guided_000.png"),
        })
        print(f"[guided {run_name}] saved")

# save experiment manifest
manifest_path = EXPERIMENT_DIR / "manifest.csv"
with manifest_path.open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)

print("\nAll done. Manifest:", manifest_path)

  0%|          | 0/30 [00:00<?, ?it/s]

[no_guidance] saved: no_guidance.png
[guided rw0.0005_ls5] saved
[guided rw0.002_ls5] saved
[guided rw0.003_ls15] saved
[guided rw0.01_ls20] saved


  0%|          | 0/30 [00:00<?, ?it/s]

[no_guidance] saved: no_guidance.png
[guided rw0.0005_ls5] saved
[guided rw0.002_ls5] saved
[guided rw0.003_ls15] saved
[guided rw0.01_ls20] saved


  0%|          | 0/30 [00:00<?, ?it/s]

[no_guidance] saved: no_guidance.png
[guided rw0.0005_ls5] saved
[guided rw0.002_ls5] saved
[guided rw0.003_ls15] saved
[guided rw0.01_ls20] saved


  0%|          | 0/30 [00:00<?, ?it/s]

[no_guidance] saved: no_guidance.png
[guided rw0.0005_ls5] saved
[guided rw0.002_ls5] saved
[guided rw0.003_ls15] saved
[guided rw0.01_ls20] saved


  0%|          | 0/30 [00:00<?, ?it/s]

[no_guidance] saved: no_guidance.png
[guided rw0.0005_ls5] saved
[guided rw0.002_ls5] saved
[guided rw0.003_ls15] saved
[guided rw0.01_ls20] saved

All done. Manifest: /content/drive/MyDrive/image_realness_project/outputs/experiment_grid/manifest.csv


In [11]:
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd

manifest = pd.read_csv(EXPERIMENT_DIR / "manifest.csv")

for prompt in PROMPTS:
    subset = manifest[manifest["prompt"] == prompt]
    n      = len(subset)

    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
    if n == 1:
        axes = [axes]

    for ax, (_, row) in zip(axes, subset.iterrows()):
        img = Image.open(row["image_path"])
        ax.imshow(img)
        if row["mode"] == "no_guidance":
            ax.set_title("No Guidance", fontsize=9)
        else:
            ax.set_title(
                f"rw={row['rationality_weight']}\nls={row['guidance_last_steps']}",
                fontsize=9,
            )
        ax.axis("off")

    plt.suptitle(f'"{prompt}"', fontsize=10, y=1.01)
    plt.tight_layout()
    plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [12]:
from PIL import Image
import torchvision.transforms.functional as TF

def score_image(path, scorer, device):
    img    = Image.open(path).convert("RGB")
    tensor = TF.to_tensor(img).unsqueeze(0).to(device)
    with torch.no_grad():
        score = scorer(tensor.float()).item()
    return score

manifest["rationality_score"] = manifest["image_path"].apply(
    lambda p: score_image(p, scorer, device)
)

manifest.to_csv(EXPERIMENT_DIR / "manifest_with_scores.csv", index=False)

# print summary
summary = manifest.groupby(["mode", "rationality_weight", "guidance_last_steps"])[
    "rationality_score"
].mean().reset_index()

print(summary.to_string(index=False))

  mode  rationality_weight  guidance_last_steps  rationality_score
guided              0.0005                  5.0           3.303946
guided              0.0020                  5.0           3.303744
guided              0.0030                 15.0           3.303589
guided              0.0100                 20.0           3.305275


In [13]:
from PIL import Image
import torchvision.transforms.functional as TF

def score_image(path):
    img = Image.open(path).convert("RGB")
    t = TF.to_tensor(img).unsqueeze(0).to(device)
    with torch.no_grad():
        return scorer(t.float()).item()

# score both
s_no_guidance = score_image(no_guidance_path)
s_guided      = score_image(guided_dir / "guided_000.png")

print(f"No Guidance:         {s_no_guidance:.4f}")
print(f"With Guidance:       {s_guided:.4f}")
print(f"Difference:          {s_guided - s_no_guidance:+.4f}")

No Guidance:         3.3298
With Guidance:       3.3283
Difference:          -0.0015


In [14]:
# score a single guided step manually
test_img = Image.open(no_guidance_path).convert("RGB")
t = TF.to_tensor(test_img).unsqueeze(0).to(device).float()
t.requires_grad_(True)

score = scorer(t).sum()
grad = torch.autograd.grad(score, t)[0]

print(f"Score:     {score.item():.4f}")
print(f"Grad mean: {grad.mean().item():.6f}")
print(f"Grad sign: {'positive' if grad.mean().item() > 0 else 'negative'}")

Score:     3.3298
Grad mean: -0.000000
Grad sign: negative


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:335.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


In [15]:
print(f"Grad norm: {grad.norm().item():.8f}")
print(f"Grad max:  {grad.abs().max().item():.8f}")

Grad norm: 0.37266785
Grad max:  0.00755307


In [16]:
import subprocess, sys, csv
from core.generation.sd_generator import generate_image
from PIL import Image
import torchvision.transforms.functional as TF
import pandas as pd
import torch

PROMPTS = [
    "a golden retriever in a sunlit meadow",
    "a bowl of fresh fruit on a wooden table",
    "a mountain lake at sunset with reflections",
    "a busy street market in the morning",
    "a cat sitting on a windowsill in rain",
]

HYPERPARAM_GRID = [
    {"rationality_weight": 0.05,  "guidance_last_steps": 10},
    {"rationality_weight": 0.1,   "guidance_last_steps": 10},
    {"rationality_weight": 0.5,   "guidance_last_steps": 10},
]

SEED           = 42
NUM_STEPS      = 30
GUIDANCE_SCALE = 7.5
EXPERIMENT_DIR = PROJECT / "outputs" / "experiment_grid_v2"
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

def score_image(path):
    img = Image.open(path).convert("RGB")
    t = TF.to_tensor(img).unsqueeze(0).to(device).float()
    with torch.no_grad():
        return scorer(t).item()

rows = []

for prompt in PROMPTS:
    prompt_slug = prompt[:40].replace(" ", "_").replace(",", "")
    prompt_dir  = EXPERIMENT_DIR / prompt_slug
    prompt_dir.mkdir(exist_ok=True)

    # no guidance
    no_guid_path = prompt_dir / "no_guidance.png"
    generate_image(pipe=pipe, prompt=prompt, output_path=no_guid_path,
                   guidance_scale=GUIDANCE_SCALE, num_inference_steps=NUM_STEPS, seed=SEED)
    rows.append({
        "prompt": prompt, "mode": "no_guidance",
        "rationality_weight": None, "guidance_last_steps": None,
        "image_path": str(no_guid_path),
        "rationality_score": score_image(no_guid_path),
    })
    print(f"[no_guidance] score: {rows[-1]['rationality_score']:.4f}")

    # guided
    for hp in HYPERPARAM_GRID:
        run_name   = f"rw{hp['rationality_weight']}_ls{hp['guidance_last_steps']}"
        guided_dir = prompt_dir / run_name
        guided_dir.mkdir(exist_ok=True)

        subprocess.run([
            sys.executable,
            str(PROJECT / "scripts" / "generate_with_rationality_guidance.py"),
            "--project",             str(PROJECT),
            "--prompt",              prompt,
            "--output-dir",          str(guided_dir),
            "--num-inference-steps", str(NUM_STEPS),
            "--guidance-scale",      str(GUIDANCE_SCALE),
            "--rationality-weight",  str(hp["rationality_weight"]),
            "--guidance-last-steps", str(hp["guidance_last_steps"]),
            "--seed",                str(SEED),
        ])

        guided_path = guided_dir / "guided_000.png"
        rows.append({
            "prompt": prompt, "mode": "guided",
            "rationality_weight": hp["rationality_weight"],
            "guidance_last_steps": hp["guidance_last_steps"],
            "image_path": str(guided_path),
            "rationality_score": score_image(guided_path),
        })
        print(f"[{run_name}] score: {rows[-1]['rationality_score']:.4f}")

# save
manifest = pd.DataFrame(rows)
manifest.to_csv(EXPERIMENT_DIR / "manifest_with_scores.csv", index=False)

# summary
summary = manifest.groupby(["mode", "rationality_weight", "guidance_last_steps"])["rationality_score"].mean()
print("\n--- Mean scores across prompts ---")
print(summary.to_string())

  0%|          | 0/30 [00:00<?, ?it/s]

[no_guidance] score: 3.3685
[rw0.05_ls10] score: 3.3713
[rw0.1_ls10] score: 3.3713
[rw0.5_ls10] score: 3.4003


  0%|          | 0/30 [00:00<?, ?it/s]

[no_guidance] score: 4.1913
[rw0.05_ls10] score: 4.1922
[rw0.1_ls10] score: 4.1930
[rw0.5_ls10] score: 4.2005


  0%|          | 0/30 [00:00<?, ?it/s]

[no_guidance] score: 3.1079
[rw0.05_ls10] score: 3.1128
[rw0.1_ls10] score: 3.1157
[rw0.5_ls10] score: 3.1419


  0%|          | 0/30 [00:00<?, ?it/s]

[no_guidance] score: 2.5231
[rw0.05_ls10] score: 2.5255
[rw0.1_ls10] score: 2.5271
[rw0.5_ls10] score: 2.5484


  0%|          | 0/30 [00:00<?, ?it/s]

[no_guidance] score: 3.3298
[rw0.05_ls10] score: 3.3302
[rw0.1_ls10] score: 3.3331
[rw0.5_ls10] score: 3.3482

--- Mean scores across prompts ---
mode    rationality_weight  guidance_last_steps
guided  0.05                10.0                   3.306396
        0.10                10.0                   3.308047
        0.50                10.0                   3.327861


In [17]:
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd

manifest = pd.read_csv(EXPERIMENT_DIR / "manifest_with_scores.csv")

for prompt in PROMPTS:
    subset = manifest[manifest["prompt"] == prompt].reset_index(drop=True)
    n = len(subset)

    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))

    for ax, (_, row) in zip(axes, subset.iterrows()):
        img = Image.open(row["image_path"])
        ax.imshow(img)
        if row["mode"] == "no_guidance":
            title = f"No Guidance\nscore: {row['rationality_score']:.4f}"
        else:
            title = f"rw={row['rationality_weight']} ls={int(row['guidance_last_steps'])}\nscore: {row['rationality_score']:.4f}"
        ax.set_title(title, fontsize=9)
        ax.axis("off")

    plt.suptitle(f'"{prompt}"', fontsize=10, y=1.01)
    plt.tight_layout()
    plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [18]:
# verify which directory images are loading from
print(manifest["image_path"].iloc[0])

/content/drive/MyDrive/image_realness_project/outputs/experiment_grid_v2/a_golden_retriever_in_a_sunlit_meadow/no_guidance.png


In [19]:
import subprocess, sys, csv
from core.generation.sd_generator import generate_image
from PIL import Image
import torchvision.transforms.functional as TF
import pandas as pd
import torch

PROMPTS = [
    "a crowded train station with motion blur",
    "a dark alley at night with a single streetlight",
    "an aerial view of a dense forest in fog",
    "a cluttered desk with papers and coffee cup",
    "a stormy ocean with crashing waves",
]

EXPERIMENT_DIR = PROJECT / "outputs" / "experiment_grid_v3"

HYPERPARAM_GRID = [
    {"rationality_weight": 0.05,  "guidance_last_steps": 10},
    {"rationality_weight": 0.1,   "guidance_last_steps": 10},
    {"rationality_weight": 0.5,   "guidance_last_steps": 10},
]

SEED           = 42
NUM_STEPS      = 30
GUIDANCE_SCALE = 7.5
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

def score_image(path):
    img = Image.open(path).convert("RGB")
    t = TF.to_tensor(img).unsqueeze(0).to(device).float()
    with torch.no_grad():
        return scorer(t).item()

rows = []

for prompt in PROMPTS:
    prompt_slug = prompt[:40].replace(" ", "_").replace(",", "")
    prompt_dir  = EXPERIMENT_DIR / prompt_slug
    prompt_dir.mkdir(exist_ok=True)

    # no guidance
    no_guid_path = prompt_dir / "no_guidance.png"
    generate_image(pipe=pipe, prompt=prompt, output_path=no_guid_path,
                   guidance_scale=GUIDANCE_SCALE, num_inference_steps=NUM_STEPS, seed=SEED)
    rows.append({
        "prompt": prompt, "mode": "no_guidance",
        "rationality_weight": None, "guidance_last_steps": None,
        "image_path": str(no_guid_path),
        "rationality_score": score_image(no_guid_path),
    })
    print(f"[no_guidance] score: {rows[-1]['rationality_score']:.4f}")

    # guided
    for hp in HYPERPARAM_GRID:
        run_name   = f"rw{hp['rationality_weight']}_ls{hp['guidance_last_steps']}"
        guided_dir = prompt_dir / run_name
        guided_dir.mkdir(exist_ok=True)

        subprocess.run([
            sys.executable,
            str(PROJECT / "scripts" / "generate_with_rationality_guidance.py"),
            "--project",             str(PROJECT),
            "--prompt",              prompt,
            "--output-dir",          str(guided_dir),
            "--num-inference-steps", str(NUM_STEPS),
            "--guidance-scale",      str(GUIDANCE_SCALE),
            "--rationality-weight",  str(hp["rationality_weight"]),
            "--guidance-last-steps", str(hp["guidance_last_steps"]),
            "--seed",                str(SEED),
        ])

        guided_path = guided_dir / "guided_000.png"
        rows.append({
            "prompt": prompt, "mode": "guided",
            "rationality_weight": hp["rationality_weight"],
            "guidance_last_steps": hp["guidance_last_steps"],
            "image_path": str(guided_path),
            "rationality_score": score_image(guided_path),
        })
        print(f"[{run_name}] score: {rows[-1]['rationality_score']:.4f}")

# save
manifest = pd.DataFrame(rows)
manifest.to_csv(EXPERIMENT_DIR / "manifest_with_scores.csv", index=False)

# summary
summary = manifest.groupby(["mode", "rationality_weight", "guidance_last_steps"])["rationality_score"].mean()
print("\n--- Mean scores across prompts ---")
print(summary.to_string())

  0%|          | 0/30 [00:00<?, ?it/s]

[no_guidance] score: 2.8471
[rw0.05_ls10] score: 2.8500
[rw0.1_ls10] score: 2.8523
[rw0.5_ls10] score: 2.8733


  0%|          | 0/30 [00:00<?, ?it/s]

[no_guidance] score: 3.7174
[rw0.05_ls10] score: 3.7192
[rw0.1_ls10] score: 3.7236
[rw0.5_ls10] score: 3.7525


  0%|          | 0/30 [00:00<?, ?it/s]

[no_guidance] score: 3.9036
[rw0.05_ls10] score: 3.9046
[rw0.1_ls10] score: 3.9066
[rw0.5_ls10] score: 3.9177


  0%|          | 0/30 [00:00<?, ?it/s]

[no_guidance] score: 3.4249
[rw0.05_ls10] score: 3.4329
[rw0.1_ls10] score: 3.4427
[rw0.5_ls10] score: 3.4839


  0%|          | 0/30 [00:00<?, ?it/s]

[no_guidance] score: 3.1556
[rw0.05_ls10] score: 3.1561
[rw0.1_ls10] score: 3.1575
[rw0.5_ls10] score: 3.1679

--- Mean scores across prompts ---
mode    rationality_weight  guidance_last_steps
guided  0.05                10.0                   3.412561
        0.10                10.0                   3.416524
        0.50                10.0                   3.439073


In [20]:
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd

manifest = pd.read_csv(EXPERIMENT_DIR / "manifest_with_scores.csv")

for prompt in PROMPTS:
    subset = manifest[manifest["prompt"] == prompt].reset_index(drop=True)
    n = len(subset)

    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))

    for ax, (_, row) in zip(axes, subset.iterrows()):
        img = Image.open(row["image_path"])
        ax.imshow(img)
        if row["mode"] == "no_guidance":
            title = f"No Guidance\nscore: {row['rationality_score']:.4f}"
        else:
            title = f"rw={row['rationality_weight']} ls={int(row['guidance_last_steps'])}\nscore: {row['rationality_score']:.4f}"
        ax.set_title(title, fontsize=9)
        ax.axis("off")

    plt.suptitle(f'"{prompt}"', fontsize=10, y=1.01)
    plt.tight_layout()
    plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [21]:
# ============================================================
# CELL 1
# Create: generate_with_rationality_guidance_no_clamp.py
# ============================================================

from pathlib import Path
PROJECT = Path("/content/drive/MyDrive/image_realness_project")
src = PROJECT/"scripts"/"generate_with_rationality_guidance.py"
dst = PROJECT/"scripts"/"generate_with_rationality_guidance_no_clamp.py"

text = src.read_text()

old = """
                    grad = torch.clamp(
                        grad,
                        -0.1,
                        0.1,
                    )
"""

text = text.replace(old, "")

dst.write_text(text)

print("Created:", dst)

Created: /content/drive/MyDrive/image_realness_project/scripts/generate_with_rationality_guidance_no_clamp.py


In [22]:
# ============================================================
# CELL 2
# Create: generate_with_rationality_guidance_early.py
# guidance_last_steps = 20
# ============================================================

from pathlib import Path
src = PROJECT/"scripts"/"generate_with_rationality_guidance.py"
dst = PROJECT/"scripts"/"generate_with_rationality_guidance_early.py"

text = src.read_text()

old = """
        default=5,
"""

new = """
        default=20,
"""

text = text.replace(old, new, 1)

dst.write_text(text)

print("Created:", dst)

Created: /content/drive/MyDrive/image_realness_project/scripts/generate_with_rationality_guidance_early.py


In [23]:
# ============================================================
# CELL 3
# Create: generate_with_rationality_guidance_scaled.py
# timestep adaptive guidance
# ============================================================

from pathlib import Path
src = PROJECT/"scripts"/"generate_with_rationality_guidance.py"
dst = PROJECT/"scripts"/"generate_with_rationality_guidance_scaled.py"

text = src.read_text()

old = """
                noise_cfg = noise_cfg - (
                    args.rationality_weight
                    * torch.sqrt(1 - alpha_bar)
                    * grad
                )
"""

new = """
                t_scale = torch.sqrt(1 - alpha_bar)

                adaptive_weight = (
                    args.rationality_weight
                    * t_scale
                    * 5.0
                )

                noise_cfg = noise_cfg - (
                    adaptive_weight * grad
                )
"""

text = text.replace(old, new)

dst.write_text(text)

print("Created:", dst)

Created: /content/drive/MyDrive/image_realness_project/scripts/generate_with_rationality_guidance_scaled.py


In [24]:
# ============================================================
# CELL 4
# Create: rationality_guidance_normalized.py
# tanh stabilized scores
# ============================================================

from pathlib import Path

src = PROJECT/"scripts"/"generate_with_rationality_guidance.py"
dst = PROJECT/"scripts"/"rationality_guidance_normalized.py"

text = src.read_text()

old = """
        score = self.joint_model.quality_R(features)
        return score
"""

new = """
        score = self.joint_model.quality_R(features)

        # stabilize score range
        score = torch.tanh(score)

        return score
"""

text = text.replace(old, new)

dst.write_text(text)

print("Created:", dst)

Created: /content/drive/MyDrive/image_realness_project/scripts/rationality_guidance_normalized.py


In [25]:
# ============================================================
# CELL 5
# Create: rationality_guidance_light_blur.py
# blur kernel 3
# ============================================================

from pathlib import Path

src = PROJECT/"scripts"/"generate_with_rationality_guidance.py"
dst = PROJECT/"scripts"/"rationality_guidance_light_blur.py"

text = src.read_text()

text = text.replace(
    "blur_kernel=11",
    "blur_kernel=3"
)

dst.write_text(text)

print("Created:", dst)

Created: /content/drive/MyDrive/image_realness_project/scripts/rationality_guidance_light_blur.py


In [26]:
# ============================================================
# CELL 6
# Create COMBINED BEST variant
# no clamp + early + adaptive
# ============================================================

from pathlib import Path

src = PROJECT/"scripts"/"generate_with_rationality_guidance.py"
dst = PROJECT/"scripts"/"generate_with_rationality_guidance_best.py"

text = src.read_text()

# ------------------------------------------------
# remove clamp
# ------------------------------------------------

old_clamp = """
                    grad = torch.clamp(
                        grad,
                        -0.1,
                        0.1,
                    )
"""

text = text.replace(old_clamp, "")

# ------------------------------------------------
# guidance steps
# ------------------------------------------------

text = text.replace(
    "default=5,",
    "default=20,",
    1
)

# ------------------------------------------------
# adaptive guidance
# ------------------------------------------------

old_guidance = """
                noise_cfg = noise_cfg - (
                    args.rationality_weight
                    * torch.sqrt(1 - alpha_bar)
                    * grad
                )
"""

new_guidance = """
                t_scale = torch.sqrt(1 - alpha_bar)

                adaptive_weight = (
                    args.rationality_weight
                    * t_scale
                    * 5.0
                )

                noise_cfg = noise_cfg - (
                    adaptive_weight * grad
                )
"""

text = text.replace(old_guidance, new_guidance)

dst.write_text(text)

print("Created:", dst)

Created: /content/drive/MyDrive/image_realness_project/scripts/generate_with_rationality_guidance_best.py


In [27]:
def save_debug_image(pipe, image_tensor, save_path):

    image_np = (
        image_tensor[0]
        .detach()
        .cpu()
        .permute(1, 2, 0)
        .float()
        .numpy()
    )

    image_np = (image_np * 255).clip(0, 255).astype("uint8")

    from PIL import Image

    image = Image.fromarray(image_np)

    image.save(save_path)

In [30]:
# ============================================================
# CELL 7
# QUICK TEST RUN
# ============================================================

from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/image_realness_project")

!python {PROJECT/"scripts"/"generate_with_rationality_guidance_best.py"} \
  --project {PROJECT} \
  --prompt "a realistic portrait photograph of a woman" \
  --output-dir {PROJECT/"outputs"/"test_best"} \
  --num-inference-steps 30 \
  --guidance-scale 7.5 \
  --rationality-weight 0.1 \
  --guidance-last-steps 20 \
  --seed 1234 \
  --save-debug-steps

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Using device: cuda:0
Loading pipeline components...:   0% 0/7 [00:00<?, ?it/s]
Loading weights:   0% 0/196 [00:00<?, ?it/s]
Loading weights:   1% 1/196 [00:00<00:00, 6955.73it/s, Materializing param=text_model.embeddings.position_embedding.weight]
Loading weights:   1% 1/196 [00:00<00:00, 3622.02it/s, Materializing param=text_model.embeddings.position_embedding.weight]
Loading weights:   1% 2/196 [00:00<00:00, 2403.61it/s, Materializing param=text_model.embeddings.token_embedding.weight]   
Loading weights:   1% 2/196 [00:00<00:00, 1563.29it/s, Materializing param=text_model.embeddings.token_embedding.weight]
Loading weights:   2% 3/196 [00:00<00:00, 1395.16it/s, Materializing param=text_model

In [31]:
# ============================================================
# CELL 8
# FAIR COMPARISON MATRIX
# same hyperparams everywhere
# ============================================================

from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/image_realness_project")

experiments = [
    ("generate_with_rationality_guidance.py", "baseline"),
    ("generate_with_rationality_guidance_no_clamp.py", "no_clamp"),
    ("generate_with_rationality_guidance_early.py", "early"),
    ("generate_with_rationality_guidance_scaled.py", "scaled"),
    ("generate_with_rationality_guidance_best.py", "best"),
]

prompt = "a realistic DSLR photo of a man sitting in a cafe"

RATIONALITY_WEIGHT = 0.1
GUIDANCE_LAST_STEPS = 20
GUIDANCE_SCALE = 7.5
NUM_STEPS = 30
SEED = 1234

for script, name in experiments:

    outdir = PROJECT / "outputs" / name

    cmd = f"""
python {PROJECT/'scripts'/script} \
  --project {PROJECT} \
  --prompt "{prompt}" \
  --output-dir {outdir} \
  --num-inference-steps {NUM_STEPS} \
  --guidance-scale {GUIDANCE_SCALE} \
  --rationality-weight {RATIONALITY_WEIGHT} \
  --guidance-last-steps {GUIDANCE_LAST_STEPS} \
  --seed {SEED} \
  --save-debug-steps
"""

    print("=" * 80)
    print(name)
    print("=" * 80)

    !$cmd

Streaming output truncated to the last 5000 lines.
Loading weights:   5% 10/196 [00:00<00:00, 949.22it/s, Materializing param=text_model.encoder.layers.0.mlp.fc2.weight] 
Loading weights:   6% 11/196 [00:00<00:00, 1012.94it/s, Materializing param=text_model.encoder.layers.0.self_attn.k_proj.bias]
Loading weights:   6% 11/196 [00:00<00:00, 997.80it/s, Materializing param=text_model.encoder.layers.0.self_attn.k_proj.bias] 
Loading weights:   6% 12/196 [00:00<00:00, 1048.20it/s, Materializing param=text_model.encoder.layers.0.self_attn.k_proj.weight]
Loading weights:   6% 12/196 [00:00<00:00, 1038.24it/s, Materializing param=text_model.encoder.layers.0.self_attn.k_proj.weight]
Loading weights:   7% 13/196 [00:00<00:00, 1031.13it/s, Materializing param=text_model.encoder.layers.0.self_attn.out_proj.bias]
Loading weights:   7% 13/196 [00:00<00:00, 1020.07it/s, Materializing param=text_model.encoder.layers.0.self_attn.out_proj.bias]
Loading weights:   7% 14/196 [00:00<00:00, 1079.58it/s, Mat